In [16]:
import importlib
import snowflake_loader

importlib.reload(snowflake_loader)

print("get_connection available:", hasattr(snowflake_loader, "get_connection"))

get_connection available: True


In [17]:
conn = snowflake_loader.get_connection()

print("Snowflake connection successful")

Snowflake connection successful


In [18]:
query = """
SELECT *
FROM YIELD_FORECAST_DB.POD_VERTEX.YIELD_FORECAST_INPUT
"""

yf_df = pd.read_sql(query, conn)

print("YIELD_FORECAST_INPUT loaded successfully.")
print("Shape:", yf_df.shape)

/tmp/ipykernel_131906/3868971005.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  yf_df = pd.read_sql(query, conn)


YIELD_FORECAST_INPUT loaded successfully.
Shape: (111, 77)


In [19]:
print("Columns:", len(yf_df.columns))
print("\nColumn names:")

for i, col in enumerate(yf_df.columns, start=1):
    print(f"{i:02d}. {col}")

Columns: 77

Column names:
01. FIELD_ID
02. SEASON_YEAR
03. WEEK_INDEX
04. SEASON_DAY
05. MEAN_TEMPERATURE
06. TEMPERATURE_7D
07. TEMPERATURE_14D
08. TEMPERATURE_DEVIATION
09. RAINFALL_7D
10. RAINFALL_14D
11. RAINFALL_DEVIATION
12. HUMIDITY_7D
13. HUMIDITY_14D
14. HUMIDITY_DEVIATION
15. LEAFWETNESS_7D
16. LEAFWETNESS_DEVIATION
17. CUMULATIVE_GDD
18. HISTORICAL_CUMULATIVE_GDD
19. WEATHER_GDD_DEVIATION
20. GDD_DEVIATION_PCT
21. WEATHER_DATA_AGE
22. WEATHER_DATA_LAG_DAYS
23. WEATHER_ALIGNMENT_STATUS
24. NDVI_VALUE
25. NDVI_LAG_DAYS
26. NDVI_FRESHNESS_STATUS
27. NDVI_VALID_FOR_FORECAST
28. DEVIATION_STATE
29. DEVIATION_LEVEL
30. DEVIATION_DIRECTION
31. DOMINANT_DEVIATION_DRIVER
32. WEATHER_DEVIATION
33. SDE_GDD_DEVIATION
34. NDVI_DEVIATION
35. PEST_DEVIATION
36. DISEASE_DEVIATION
37. SIGNALS_ABNORMAL_COUNT
38. SIGNALS_CONFIRMING_COUNT
39. MULTI_SIGNAL_CONFIRMED
40. DEVIATION_CONFIDENCE
41. RISK_MISSING_SIGNAL_COUNT
42. MODEL_VERSION
43. PEST_RISK_SCORE
44. GDD_RISK_SCORE
45. WEATHER_DISEAS

In [20]:
target_check = yf_df[
    ["ACTUAL_YIELD_BU_ACRE", "ACTUAL_YIELD_NUMERIC", "YIELD_TARGET_STATUS"]
].copy()

print("Target status:")
print(yf_df["YIELD_TARGET_STATUS"].value_counts(dropna=False))

print("\nMissing numeric targets:")
print(yf_df["ACTUAL_YIELD_NUMERIC"].isna().sum())

print("\nNumeric target summary:")
print(yf_df["ACTUAL_YIELD_NUMERIC"].describe())

Target status:
YIELD_TARGET_STATUS
VALID    111
Name: count, dtype: int64

Missing numeric targets:
0

Numeric target summary:
count    111.000000
mean     151.315315
std       69.034969
min       58.000000
25%       63.500000
50%      182.000000
75%      209.000000
max      218.000000
Name: ACTUAL_YIELD_NUMERIC, dtype: float64


In [21]:
role_columns = [
    col for col in yf_df.columns
    if "ROLE" in col.upper()
    or "FEATURE" in col.upper()
    or "CANDIDATE" in col.upper()
]

print("Potential role/feature columns:")
print(role_columns)

Potential role/feature columns:
[]


In [26]:
feature_dict = pd.read_csv("YIELD_FORECAST_FEATURE_DICTIONARY.csv")

print("Feature Dictionary loaded successfully.")
print("Shape:", feature_dict.shape)

Feature Dictionary loaded successfully.
Shape: (77, 10)


In [27]:
feature_dict.head()

,COLUMN_NAME,DATA_TYPE,MISSING_COUNT,UNIQUE_VALUES,COLUMN_ROLE,SOURCE,DESCRIPTION,FORECAST_AVAILABILITY,MISSING_VALUE_HANDLING,LEAKAGE_RULE
0,FIELD_ID,object,0,5,FORECAST_KEY,FIELD_SEASON_CONTEXT,Unique field identifier for the forecast record.,Available at forecast time.,Required; missing values are not permitted.,Identifier only; contains no future target inf...
1,SEASON_YEAR,int64,0,4,FORECAST_KEY,FIELD_SEASON_CONTEXT,Crop season year associated with the field.,Available at forecast time.,Required; missing values are not permitted.,Season identifier only.
2,WEEK_INDEX,int64,0,6,FORECAST_KEY,SEASON_INTELLIGENCE_INPUT,Canonical seasonal week index used for forecas...,Available at forecast time.,Required; missing values are not permitted.,Derived from the established seasonal timeline...
3,SEASON_DAY,int64,0,42,FORECAST_KEY,SEASON_INTELLIGENCE_INPUT,Canonical seasonal day corresponding to the fo...,Available at forecast time.,Required; missing values are not permitted.,Derived from the seasonal timeline only.
4,MEAN_TEMPERATURE,float64,5,81,MODEL_CANDIDATE,NaN,NaN,NaN,NaN,NaN


In [28]:
role_summary = (
    feature_dict["COLUMN_ROLE"]
    .value_counts()
    .rename_axis("COLUMN_ROLE")
    .reset_index(name="COUNT")
)

print(role_summary)

          COLUMN_ROLE  COUNT
0     MODEL_CANDIDATE     57
1  PROVENANCE_QUALITY     11
2        FORECAST_KEY      4
3              TARGET      3
4   CONSTANT_METADATA      2


In [29]:
model_features = feature_dict.loc[
    feature_dict["COLUMN_ROLE"] == "MODEL_CANDIDATE",
    "COLUMN_NAME"
].tolist()

print("Number of model features:", len(model_features))

print("\nModel features:")
for i, feature in enumerate(model_features, start=1):
    print(f"{i:02d}. {feature}")

Number of model features: 57

Model features:
01. MEAN_TEMPERATURE
02. TEMPERATURE_7D
03. TEMPERATURE_14D
04. TEMPERATURE_DEVIATION
05. RAINFALL_7D
06. RAINFALL_14D
07. RAINFALL_DEVIATION
08. HUMIDITY_7D
09. HUMIDITY_14D
10. HUMIDITY_DEVIATION
11. LEAFWETNESS_7D
12. LEAFWETNESS_DEVIATION
13. CUMULATIVE_GDD
14. HISTORICAL_CUMULATIVE_GDD
15. WEATHER_GDD_DEVIATION
16. GDD_DEVIATION_PCT
17. NDVI_VALUE
18. DEVIATION_STATE
19. DEVIATION_LEVEL
20. DEVIATION_DIRECTION
21. DOMINANT_DEVIATION_DRIVER
22. WEATHER_DEVIATION
23. SDE_GDD_DEVIATION
24. NDVI_DEVIATION
25. PEST_DEVIATION
26. DISEASE_DEVIATION
27. SIGNALS_ABNORMAL_COUNT
28. SIGNALS_CONFIRMING_COUNT
29. MULTI_SIGNAL_CONFIRMED
30. DEVIATION_CONFIDENCE
31. PEST_RISK_SCORE
32. GDD_RISK_SCORE
33. WEATHER_DISEASE_RISK_SCORE
34. NDVI_STRESS_SCORE
35. HISTORICAL_DEVIATION_RISK_SCORE
36. OVERALL_RISK_SCORE
37. OVERALL_RISK_LOWER_BOUND
38. OVERALL_RISK_UPPER_BOUND
39. RISK_LEVEL
40. IS_PROVISIONAL
41. AVAILABLE_WEIGHT_PCT
42. DOMINANT_RISK_DRIVER


In [30]:
feature_dtypes = yf_df[model_features].dtypes

print("Model feature data types:")
print(feature_dtypes)

print("\nData type summary:")
print(feature_dtypes.value_counts())

Model feature data types:
MEAN_TEMPERATURE                   float64
TEMPERATURE_7D                     float64
TEMPERATURE_14D                    float64
TEMPERATURE_DEVIATION              float64
RAINFALL_7D                        float64
RAINFALL_14D                       float64
RAINFALL_DEVIATION                 float64
HUMIDITY_7D                        float64
HUMIDITY_14D                       float64
HUMIDITY_DEVIATION                 float64
LEAFWETNESS_7D                     float64
LEAFWETNESS_DEVIATION              float64
CUMULATIVE_GDD                     float64
HISTORICAL_CUMULATIVE_GDD          float64
WEATHER_GDD_DEVIATION              float64
GDD_DEVIATION_PCT                  float64
NDVI_VALUE                         float64
DEVIATION_STATE                     object
DEVIATION_LEVEL                     object
DEVIATION_DIRECTION                 object
DOMINANT_DEVIATION_DRIVER           object
WEATHER_DEVIATION                  float64
SDE_GDD_DEVIATION           

In [31]:
target_column = "ACTUAL_YIELD_NUMERIC"

X = yf_df[model_features].copy()
y = yf_df[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget:")
print(target_column)

print("\nCategorical features:")
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
print(categorical_features)

X shape: (111, 57)
y shape: (111,)

Target:
ACTUAL_YIELD_NUMERIC

Categorical features:
['DEVIATION_STATE', 'DEVIATION_LEVEL', 'DEVIATION_DIRECTION', 'DOMINANT_DEVIATION_DRIVER', 'DEVIATION_CONFIDENCE', 'RISK_LEVEL', 'DOMINANT_RISK_DRIVER', 'CONFIDENCE_LEVEL', 'CROP_TYPE']


In [32]:
missing_features = X.isna().sum()

missing_features = missing_features[missing_features > 0].sort_values(ascending=False)

print("Features with missing values:")
print(missing_features)

print("\nTotal missing feature values:", X.isna().sum().sum())


Features with missing values:
GDD_RISK_SCORE                     13
MEAN_TEMPERATURE                    5
TEMPERATURE_14D                     5
TEMPERATURE_DEVIATION               5
RAINFALL_7D                         5
TEMPERATURE_7D                      5
RAINFALL_14D                        5
RAINFALL_DEVIATION                  5
HUMIDITY_14D                        5
HUMIDITY_7D                         5
LEAFWETNESS_7D                      5
LEAFWETNESS_DEVIATION               5
CUMULATIVE_GDD                      5
HUMIDITY_DEVIATION                  5
HISTORICAL_CUMULATIVE_GDD           5
WEATHER_GDD_DEVIATION               5
NDVI_VALUE                          5
GDD_DEVIATION_PCT                   5
NDVI_STRESS_SCORE                   4
PEST_RISK_SCORE                     3
WEATHER_DISEASE_RISK_SCORE          1
HISTORICAL_DEVIATION_RISK_SCORE     1
dtype: int64

Total missing feature values: 107


In [33]:
missing_feature_rules = feature_dict[
    feature_dict["COLUMN_NAME"].isin(missing_features.index)
][
    [
        "COLUMN_NAME",
        "COLUMN_ROLE",
        "MISSING_COUNT",
        "MISSING_VALUE_HANDLING",
        "FORECAST_AVAILABILITY",
        "LEAKAGE_RULE"
    ]
].sort_values("MISSING_COUNT", ascending=False)

missing_feature_rules

,COLUMN_NAME,COLUMN_ROLE,MISSING_COUNT,MISSING_VALUE_HANDLING,FORECAST_AVAILABILITY,LEAKAGE_RULE
43,GDD_RISK_SCORE,MODEL_CANDIDATE,13,NaN,NaN,NaN
4,MEAN_TEMPERATURE,MODEL_CANDIDATE,5,NaN,NaN,NaN
6,TEMPERATURE_14D,MODEL_CANDIDATE,5,NaN,NaN,NaN
7,TEMPERATURE_DEVIATION,MODEL_CANDIDATE,5,NaN,NaN,NaN
8,RAINFALL_7D,MODEL_CANDIDATE,5,NaN,NaN,NaN
5,TEMPERATURE_7D,MODEL_CANDIDATE,5,NaN,NaN,NaN
9,RAINFALL_14D,MODEL_CANDIDATE,5,NaN,NaN,NaN
10,RAINFALL_DEVIATION,MODEL_CANDIDATE,5,NaN,NaN,NaN
12,HUMIDITY_14D,MODEL_CANDIDATE,5,NaN,NaN,NaN
11,HUMIDITY_7D,MODEL_CANDIDATE,5,NaN,NaN,NaN


In [34]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical features:")
for feature in categorical_features:
    print(f"- {feature}: {X[feature].isna().sum()} missing")

Categorical features:
- DEVIATION_STATE: 0 missing
- DEVIATION_LEVEL: 0 missing
- DEVIATION_DIRECTION: 0 missing
- DOMINANT_DEVIATION_DRIVER: 0 missing
- DEVIATION_CONFIDENCE: 0 missing
- RISK_LEVEL: 0 missing
- DOMINANT_RISK_DRIVER: 0 missing
- CONFIDENCE_LEVEL: 0 missing
- CROP_TYPE: 0 missing


In [35]:
non_float_features = X.select_dtypes(
    exclude=["float64"]
).columns.tolist()

print("Non-float feature missing values:")
print(X[non_float_features].isna().sum())

Non-float feature missing values:
DEVIATION_STATE              0
DEVIATION_LEVEL              0
DEVIATION_DIRECTION          0
DOMINANT_DEVIATION_DRIVER    0
SIGNALS_ABNORMAL_COUNT       0
SIGNALS_CONFIRMING_COUNT     0
MULTI_SIGNAL_CONFIRMED       0
DEVIATION_CONFIDENCE         0
RISK_LEVEL                   0
IS_PROVISIONAL               0
DOMINANT_RISK_DRIVER         0
CONFLICTING_SIGNALS_FLAG     0
CONFIDENCE_LEVEL             0
AVAILABLE_SIGNAL_COUNT       0
SDE_MISSING_SIGNAL_COUNT     0
CROP_TYPE                    0
HISTORICAL_YIELD_COUNT       0
dtype: int64


In [36]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numeric features: 45
Categorical features: 9

Numeric features:
['MEAN_TEMPERATURE', 'TEMPERATURE_7D', 'TEMPERATURE_14D', 'TEMPERATURE_DEVIATION', 'RAINFALL_7D', 'RAINFALL_14D', 'RAINFALL_DEVIATION', 'HUMIDITY_7D', 'HUMIDITY_14D', 'HUMIDITY_DEVIATION', 'LEAFWETNESS_7D', 'LEAFWETNESS_DEVIATION', 'CUMULATIVE_GDD', 'HISTORICAL_CUMULATIVE_GDD', 'WEATHER_GDD_DEVIATION', 'GDD_DEVIATION_PCT', 'NDVI_VALUE', 'WEATHER_DEVIATION', 'SDE_GDD_DEVIATION', 'NDVI_DEVIATION', 'PEST_DEVIATION', 'DISEASE_DEVIATION', 'SIGNALS_ABNORMAL_COUNT', 'SIGNALS_CONFIRMING_COUNT', 'PEST_RISK_SCORE', 'GDD_RISK_SCORE', 'WEATHER_DISEASE_RISK_SCORE', 'NDVI_STRESS_SCORE', 'HISTORICAL_DEVIATION_RISK_SCORE', 'OVERALL_RISK_SCORE', 'OVERALL_RISK_LOWER_BOUND', 'OVERALL_RISK_UPPER_BOUND', 'AVAILABLE_WEIGHT_PCT', 'CONF_DATA_COMPLETENESS', 'CONF_PEST_FRESHNESS', 'CONF_NDVI_QUALITY', 'CONF_DISEASE_CONFIDENCE', 'CONFIDENCE_SCORE', 'AVAILABLE_SIGNAL_COUNT', 'SDE_MISSING_SIGNAL_COUNT', 'PREVIOUS_SEASON_YIELD', 'HISTORICAL_MEAN_YIELD'

In [37]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Preprocessor created successfully.
Numeric features: 45
Categorical features: 9


In [38]:
X_processed = preprocessor.fit_transform(X)

print("Preprocessing completed successfully.")
print("Original X shape:", X.shape)
print("Processed X shape:", X_processed.shape)

print("\nRemaining missing values:",
      pd.isna(X_processed).sum())

print("\nProcessed data type:",
      type(X_processed))

Preprocessing completed successfully.
Original X shape: (111, 57)
Processed X shape: (111, 77)

Remaining missing values: 0

Processed data type: <class 'numpy.ndarray'>


In [39]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))

print("\nFirst 20 processed feature names:")
print(processed_feature_names[:20])

Number of processed features: 77

First 20 processed feature names:
['numeric__MEAN_TEMPERATURE' 'numeric__TEMPERATURE_7D'
 'numeric__TEMPERATURE_14D' 'numeric__TEMPERATURE_DEVIATION'
 'numeric__RAINFALL_7D' 'numeric__RAINFALL_14D'
 'numeric__RAINFALL_DEVIATION' 'numeric__HUMIDITY_7D'
 'numeric__HUMIDITY_14D' 'numeric__HUMIDITY_DEVIATION'
 'numeric__LEAFWETNESS_7D' 'numeric__LEAFWETNESS_DEVIATION'
 'numeric__CUMULATIVE_GDD' 'numeric__HISTORICAL_CUMULATIVE_GDD'
 'numeric__WEATHER_GDD_DEVIATION' 'numeric__GDD_DEVIATION_PCT'
 'numeric__NDVI_VALUE' 'numeric__WEATHER_DEVIATION'
 'numeric__SDE_GDD_DEVIATION' 'numeric__NDVI_DEVIATION']


In [40]:
baseline_features = [
    "PREVIOUS_SEASON_YIELD",
    "HISTORICAL_MEAN_YIELD",
    "HISTORICAL_YIELD_STD",
    "HISTORICAL_YIELD_COUNT",
    "HISTORICAL_YIELD_TREND"
]

X_baseline = yf_df[baseline_features].copy()

print("Baseline feature shape:", X_baseline.shape)
print("\nBaseline features:")
print(baseline_features)

print("\nMissing values:")
print(X_baseline.isna().sum())


Baseline feature shape: (111, 5)

Baseline features:
['PREVIOUS_SEASON_YIELD', 'HISTORICAL_MEAN_YIELD', 'HISTORICAL_YIELD_STD', 'HISTORICAL_YIELD_COUNT', 'HISTORICAL_YIELD_TREND']

Missing values:
PREVIOUS_SEASON_YIELD     0
HISTORICAL_MEAN_YIELD     0
HISTORICAL_YIELD_STD      0
HISTORICAL_YIELD_COUNT    0
HISTORICAL_YIELD_TREND    0
dtype: int64


In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_baseline_mean = X_baseline["HISTORICAL_MEAN_YIELD"]

mae_mean = mean_absolute_error(y, y_baseline_mean)
rmse_mean = np.sqrt(mean_squared_error(y, y_baseline_mean))
r2_mean = r2_score(y, y_baseline_mean)

print("Historical Mean Baseline")
print("------------------------")
print("MAE :", round(mae_mean, 3))
print("RMSE:", round(rmse_mean, 3))
print("R²  :", round(r2_mean, 3))

Historical Mean Baseline
------------------------
MAE : 4.554
RMSE: 5.87
R²  : 0.993


In [42]:
y_baseline_previous = X_baseline["PREVIOUS_SEASON_YIELD"]

mae_previous = mean_absolute_error(y, y_baseline_previous)
rmse_previous = np.sqrt(mean_squared_error(y, y_baseline_previous))
r2_previous = r2_score(y, y_baseline_previous)

print("Previous Season Yield Baseline")
print("------------------------------")
print("MAE :", round(mae_previous, 3))
print("RMSE:", round(rmse_previous, 3))
print("R²  :", round(r2_previous, 3))

Previous Season Yield Baseline
------------------------------
MAE : 4.437
RMSE: 5.097
R²  : 0.995


In [43]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_baseline, y)

y_ridge = ridge_model.predict(X_baseline)

mae_ridge = mean_absolute_error(y, y_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y, y_ridge))
r2_ridge = r2_score(y, y_ridge)

print("Ridge Regression Baseline")
print("------------------------")
print("MAE :", round(mae_ridge, 3))
print("RMSE:", round(rmse_ridge, 3))
print("R²  :", round(r2_ridge, 3))

Ridge Regression Baseline
------------------------
MAE : 3.151
RMSE: 3.912
R²  : 0.997


In [44]:
baseline_results = pd.DataFrame({
    "MODEL": [
        "Historical Mean",
        "Previous Season Yield",
        "Ridge Regression"
    ],
    "MAE": [
        mae_mean,
        mae_previous,
        mae_ridge
    ],
    "RMSE": [
        rmse_mean,
        rmse_previous,
        rmse_ridge
    ],
    "R2": [
        r2_mean,
        r2_previous,
        r2_ridge
    ]
})

baseline_results

,MODEL,MAE,RMSE,R2
0,Historical Mean,4.554054,5.870371,0.992703
1,Previous Season Yield,4.436937,5.096590,0.994500
2,Ridge Regression,3.151145,3.911994,0.996760


In [47]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.2.0


In [48]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("XGBoost model created successfully.")
print(xgb_model)

XGBoost model created successfully.
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)


In [49]:
xgb_model.fit(X_processed, y)

y_xgb = xgb_model.predict(X_processed)

mae_xgb = mean_absolute_error(y, y_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y, y_xgb))
r2_xgb = r2_score(y, y_xgb)

print("Initial XGBoost Training Results")
print("--------------------------------")
print("MAE :", round(mae_xgb, 3))
print("RMSE:", round(rmse_xgb, 3))
print("R²  :", round(r2_xgb, 3))

Initial XGBoost Training Results
--------------------------------
MAE : 0.006
RMSE: 0.008
R²  : 1.0


In [50]:
print("Available seasons:")
print(sorted(yf_df["SEASON_YEAR"].unique()))

print("\nRows by season:")
print(yf_df["SEASON_YEAR"].value_counts().sort_index())

Available seasons:
[2023, 2024, 2025, 2026]

Rows by season:
SEASON_YEAR
2023    30
2024    29
2025    30
2026    22
Name: count, dtype: int64


In [51]:
backtest_folds = [
    {
        "fold": 1,
        "train_seasons": [2023],
        "test_season": 2024
    },
    {
        "fold": 2,
        "train_seasons": [2023, 2024],
        "test_season": 2025
    },
    {
        "fold": 3,
        "train_seasons": [2023, 2024, 2025],
        "test_season": 2026
    }
]

for fold in backtest_folds:
    print(
        f"Fold {fold['fold']}: "
        f"Train={fold['train_seasons']} → "
        f"Test={fold['test_season']}"
    )

Fold 1: Train=[2023] → Test=2024
Fold 2: Train=[2023, 2024] → Test=2025
Fold 3: Train=[2023, 2024, 2025] → Test=2026


In [52]:
for fold in backtest_folds:
    train_rows = yf_df[
        yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    ].shape[0]

    test_rows = yf_df[
        yf_df["SEASON_YEAR"] == fold["test_season"]
    ].shape[0]

    print(
        f"Fold {fold['fold']}: "
        f"Train rows = {train_rows}, "
        f"Test rows = {test_rows}"
    )

Fold 1: Train rows = 30, Test rows = 29
Fold 2: Train rows = 59, Test rows = 30
Fold 3: Train rows = 89, Test rows = 22


In [53]:
for fold in backtest_folds:
    train_rows = yf_df[
        yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    ].shape[0]

    test_rows = yf_df[
        yf_df["SEASON_YEAR"] == fold["test_season"]
    ].shape[0]

    print(
        f"Fold {fold['fold']}: "
        f"Train rows = {train_rows}, "
        f"Test rows = {test_rows}"
    )

Fold 1: Train rows = 30, Test rows = 29
Fold 2: Train rows = 59, Test rows = 30
Fold 3: Train rows = 89, Test rows = 22


In [54]:
backtest_results = []

for fold in backtest_folds:
    train_mask = yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    test_mask = yf_df["SEASON_YEAR"] == fold["test_season"]

    X_train = yf_df.loc[train_mask, model_features].copy()
    y_train = yf_df.loc[train_mask, target_column].copy()

    X_test = yf_df.loc[test_mask, model_features].copy()
    y_test = yf_df.loc[test_mask, target_column].copy()

    # Fit preprocessing ONLY on training data
    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ]
    )

    X_train_processed = fold_preprocessor.fit_transform(X_train)
    X_test_processed = fold_preprocessor.transform(X_test)

    # Fresh XGBoost model for each fold
    fold_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_train_processed, y_train)

    y_pred = fold_model.predict(X_test_processed)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    backtest_results.append({
        "FOLD": fold["fold"],
        "TRAIN_SEASONS": str(fold["train_seasons"]),
        "TEST_SEASON": fold["test_season"],
        "TRAIN_ROWS": len(X_train),
        "TEST_ROWS": len(X_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

backtest_results_df = pd.DataFrame(backtest_results)

backtest_results_df

,FOLD,TRAIN_SEASONS,TEST_SEASON,TRAIN_ROWS,TEST_ROWS,MAE,RMSE,R2
0,1,[2023],2024,30,29,84.406894,106.087305,-1.194394
1,2,"[2023, 2024]",2025,59,30,3.649722,4.425174,0.996018
2,3,"[2023, 2024, 2025]",2026,89,22,4.851672,5.540078,0.990805


In [55]:
season_target_summary = (
    yf_df
    .groupby("SEASON_YEAR")["ACTUAL_YIELD_NUMERIC"]
    .agg(["count", "min", "max", "mean", "std"])
    .round(3)
)

season_target_summary

,count,min,max,mean,std
SEASON_YEAR,,,,,
2023,30,58.0,209.0,142.000,70.827
2024,29,61.0,218.0,151.000,72.883
2025,30,63.5,214.0,148.600,71.328
2026,22,64.0,209.0,168.136,59.134


In [56]:
train_mask = yf_df["SEASON_YEAR"] == 2023
test_mask = yf_df["SEASON_YEAR"] == 2024

X_train = yf_df.loc[train_mask, model_features].copy()
y_train = yf_df.loc[train_mask, target_column].copy()

X_test = yf_df.loc[test_mask, model_features].copy()
y_test = yf_df.loc[test_mask, target_column].copy()

fold1_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X_train_processed = fold1_preprocessor.fit_transform(X_train)
X_test_processed = fold1_preprocessor.transform(X_test)

fold1_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

fold1_model.fit(X_train_processed, y_train)

fold1_pred = fold1_model.predict(X_test_processed)

fold1_predictions = pd.DataFrame({
    "SEASON_YEAR": yf_df.loc[test_mask, "SEASON_YEAR"].values,
    "ACTUAL_YIELD": y_test.values,
    "PREDICTED_YIELD": fold1_pred
})

fold1_predictions["ERROR"] = (
    fold1_predictions["PREDICTED_YIELD"]
    - fold1_predictions["ACTUAL_YIELD"]
)

fold1_predictions["ABS_ERROR"] = (
    fold1_predictions["ERROR"].abs()
)

fold1_predictions.sort_values(
    "ABS_ERROR",
    ascending=False
)

,SEASON_YEAR,ACTUAL_YIELD,PREDICTED_YIELD,ERROR,ABS_ERROR
5,2024,218.0,68.461647,-149.538353,149.538353
4,2024,218.0,69.351318,-148.648682,148.648682
2,2024,218.0,72.202187,-145.797813,145.797813
3,2024,218.0,72.264786,-145.735214,145.735214
23,2024,218.0,72.839020,-145.160980,145.160980
1,2024,218.0,73.035347,-144.964653,144.964653
28,2024,218.0,75.847595,-142.152405,142.152405
0,2024,218.0,77.507774,-140.492226,140.492226
26,2024,218.0,79.836716,-138.163284,138.163284
25,2024,218.0,80.017555,-137.982445,137.982445


In [58]:
fold1_comparison = yf_df.loc[
    yf_df["SEASON_YEAR"].isin([2023, 2024]),
    [
        "SEASON_YEAR",
        "PREVIOUS_SEASON_YIELD",
        "HISTORICAL_MEAN_YIELD",
        "HISTORICAL_YIELD_STD",
        "HISTORICAL_YIELD_COUNT",
        "HISTORICAL_YIELD_TREND",
        "ACTUAL_YIELD_NUMERIC"
    ]
].sort_values(
    ["SEASON_YEAR", "ACTUAL_YIELD_NUMERIC"]
)

fold1_comparison

,SEASON_YEAR,PREVIOUS_SEASON_YIELD,HISTORICAL_MEAN_YIELD,HISTORICAL_YIELD_STD,HISTORICAL_YIELD_COUNT,HISTORICAL_YIELD_TREND,ACTUAL_YIELD_NUMERIC
23,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
24,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
25,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
26,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
27,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
28,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
69,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
70,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
71,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0
72,2023,58.5,60.250000,2.474874,2,-3.500000e+00,58.0


In [59]:
train_2023 = yf_df[yf_df["SEASON_YEAR"] == 2023].copy()

print(
    train_2023[
        [
            "HISTORICAL_MEAN_YIELD",
            "PREVIOUS_SEASON_YIELD",
            "ACTUAL_YIELD_NUMERIC"
        ]
    ].sort_values("ACTUAL_YIELD_NUMERIC")
)

    HISTORICAL_MEAN_YIELD  PREVIOUS_SEASON_YIELD  ACTUAL_YIELD_NUMERIC
23                  60.25                   58.5                  58.0
24                  60.25                   58.5                  58.0
26                  60.25                   58.5                  58.0
25                  60.25                   58.5                  58.0
28                  60.25                   58.5                  58.0
27                  60.25                   58.5                  58.0
69                  60.25                   58.5                  58.0
70                  60.25                   58.5                  58.0
71                  60.25                   58.5                  58.0
72                  60.25                   58.5                  58.0
74                  60.25                   58.5                  58.0
73                  60.25                   58.5                  58.0
47                 178.50                  181.0                 176.0
48    

In [60]:
ridge_backtest_results = []

for fold in backtest_folds:
    train_mask = yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    test_mask = yf_df["SEASON_YEAR"] == fold["test_season"]

    X_train = yf_df.loc[train_mask, baseline_features].copy()
    y_train = yf_df.loc[train_mask, target_column].copy()

    X_test = yf_df.loc[test_mask, baseline_features].copy()
    y_test = yf_df.loc[test_mask, target_column].copy()

    ridge_fold = Ridge(alpha=1.0)
    ridge_fold.fit(X_train, y_train)

    y_pred = ridge_fold.predict(X_test)

    ridge_backtest_results.append({
        "FOLD": fold["fold"],
        "TRAIN_SEASONS": str(fold["train_seasons"]),
        "TEST_SEASON": fold["test_season"],
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

ridge_backtest_results_df = pd.DataFrame(ridge_backtest_results)

ridge_backtest_results_df

,FOLD,TRAIN_SEASONS,TEST_SEASON,MAE,RMSE,R2
0,1,[2023],2024,9.117450,10.276995,0.979407
1,2,"[2023, 2024]",2025,4.793246,6.989250,0.990067
2,3,"[2023, 2024, 2025]",2026,11.867187,13.623670,0.944395


In [61]:
feature_distribution = yf_df.groupby("SEASON_YEAR")[numeric_features].mean().T

feature_distribution["RANGE_2023_2024"] = (
    feature_distribution[2024] - feature_distribution[2023]
).abs()

feature_distribution.sort_values(
    "RANGE_2023_2024",
    ascending=False
).head(15)

SEASON_YEAR,2023,2024,2025,2026,RANGE_2023_2024
SDE_GDD_DEVIATION,-30.286667,130.462069,-33.766667,-38.463636,160.748736
HISTORICAL_CUMULATIVE_GDD,524.601484,599.720690,523.891379,513.266667,75.119206
CUMULATIVE_GDD,524.138462,576.641379,471.337931,498.547727,52.502918
NDVI_STRESS_SCORE,32.687510,0.880128,0.000000,0.111111,31.807382
WEATHER_GDD_DEVIATION,-0.463022,-23.079310,-52.553448,-14.718939,22.616288
PEST_DEVIATION,18.316667,8.696552,-17.931667,-20.150000,9.620115
OVERALL_RISK_SCORE,33.230283,25.590579,30.223150,27.464427,7.639704
OVERALL_RISK_UPPER_BOUND,33.230283,25.590579,30.223150,44.956514,7.639704
OVERALL_RISK_LOWER_BOUND,33.230283,25.590579,30.223150,22.001968,7.639704
PEST_RISK_SCORE,63.720483,56.352583,55.518417,30.824000,7.367901


In [62]:
for feature in [
    "SDE_GDD_DEVIATION",
    "HISTORICAL_CUMULATIVE_GDD",
    "CUMULATIVE_GDD",
    "NDVI_STRESS_SCORE",
    "WEATHER_GDD_DEVIATION"
]:
    print(f"\n{feature}")
    print(
        yf_df.groupby("SEASON_YEAR")[feature]
        .agg(["min", "max", "mean"])
        .round(3)
    )


SDE_GDD_DEVIATION
                min    max     mean
SEASON_YEAR                        
2023         -95.70   86.1  -30.287
2024          45.70  237.3  130.462
2025        -164.60    0.0  -33.767
2026        -451.35  286.9  -38.464

HISTORICAL_CUMULATIVE_GDD
                min      max     mean
SEASON_YEAR                          
2023         105.70  1299.05  524.601
2024          49.70  1357.30  599.721
2025          41.35  1307.50  523.891
2026          41.90  1266.70  513.267

CUMULATIVE_GDD
              min     max     mean
SEASON_YEAR                       
2023         93.9  1354.6  524.138
2024         55.7  1310.7  576.641
2025         39.9  1185.1  471.338
2026          8.2  1360.7  498.548

NDVI_STRESS_SCORE
               min     max    mean
SEASON_YEAR                       
2023         1.072  60.000  32.688
2024         0.000   8.291   0.880
2025         0.000   0.000   0.000
2026         0.000   2.000   0.111

WEATHER_GDD_DEVIATION
                 min      max   

In [63]:
train_2023 = yf_df[yf_df["SEASON_YEAR"] == 2023]
test_2024 = yf_df[yf_df["SEASON_YEAR"] == 2024]

out_of_range_results = []

for feature in numeric_features:
    train_min = train_2023[feature].min()
    train_max = train_2023[feature].max()

    out_of_range = (
        (test_2024[feature] < train_min) |
        (test_2024[feature] > train_max)
    ).sum()

    out_of_range_results.append({
        "FEATURE": feature,
        "TRAIN_2023_MIN": train_min,
        "TRAIN_2023_MAX": train_max,
        "TEST_2024_OUT_OF_RANGE_COUNT": out_of_range,
        "TEST_2024_ROWS": len(test_2024)
    })

out_of_range_df = (
    pd.DataFrame(out_of_range_results)
    .sort_values(
        "TEST_2024_OUT_OF_RANGE_COUNT",
        ascending=False
    )
)

out_of_range_df.head(20)

,FEATURE,TRAIN_2023_MIN,TRAIN_2023_MAX,TEST_2024_OUT_OF_RANGE_COUNT,TEST_2024_ROWS
43,HISTORICAL_YIELD_COUNT,2.000000,2.000000,29,29
27,NDVI_STRESS_SCORE,1.071800,60.000000,24,29
41,HISTORICAL_MEAN_YIELD,60.250000,203.000000,23,29
18,SDE_GDD_DEVIATION,-95.700000,86.100000,20,29
17,WEATHER_DEVIATION,-2.200000,1.400000,16,29
8,HUMIDITY_14D,44.600000,58.900000,11,29
40,PREVIOUS_SEASON_YIELD,58.500000,214.000000,11,29
42,HISTORICAL_YIELD_STD,2.474874,15.556349,11,29
5,RAINFALL_14D,0.110000,0.790000,11,29
9,HUMIDITY_DEVIATION,-8.100000,6.587500,10,29


In [64]:
categorical_shift = []

for feature in categorical_features:
    train_values = set(
        yf_df.loc[
            yf_df["SEASON_YEAR"] == 2023,
            feature
        ].dropna().unique()
    )

    test_values = set(
        yf_df.loc[
            yf_df["SEASON_YEAR"] == 2024,
            feature
        ].dropna().unique()
    )

    unseen_values = test_values - train_values

    categorical_shift.append({
        "FEATURE": feature,
        "TRAIN_2023_LEVELS": len(train_values),
        "TEST_2024_LEVELS": len(test_values),
        "UNSEEN_2024_LEVELS": len(unseen_values),
        "UNSEEN_VALUES": sorted(unseen_values)
    })

categorical_shift_df = pd.DataFrame(categorical_shift)

categorical_shift_df

,FEATURE,TRAIN_2023_LEVELS,TEST_2024_LEVELS,UNSEEN_2024_LEVELS,UNSEEN_VALUES
0,DEVIATION_STATE,3,3,0,[]
1,DEVIATION_LEVEL,2,3,1,[HIGH]
2,DEVIATION_DIRECTION,4,2,0,[]
3,DOMINANT_DEVIATION_DRIVER,3,5,2,"[GDD, WEATHER]"
4,DEVIATION_CONFIDENCE,2,3,1,[HIGH]
5,RISK_LEVEL,2,2,0,[]
6,DOMINANT_RISK_DRIVER,3,2,0,[]
7,CONFIDENCE_LEVEL,2,2,0,[]
8,CROP_TYPE,2,2,0,[]


In [65]:
print("XGBoost Time-Aware Backtesting Summary")
print("---------------------------------------")

print(
    backtest_results_df[
        ["TEST_SEASON", "MAE", "RMSE", "R2"]
    ].round(3)
)

print("\nMean across folds:")
print(
    backtest_results_df[
        ["MAE", "RMSE", "R2"]
    ].mean().round(3)
)

XGBoost Time-Aware Backtesting Summary
---------------------------------------
   TEST_SEASON     MAE     RMSE     R2
0         2024  84.407  106.087 -1.194
1         2025   3.650    4.425  0.996
2         2026   4.852    5.540  0.991

Mean across folds:
MAE     30.969
RMSE    38.684
R2       0.264
dtype: float64


In [66]:
week_results = []

for fold in backtest_folds:
    train_mask = yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    test_mask = yf_df["SEASON_YEAR"] == fold["test_season"]

    X_train = yf_df.loc[train_mask, model_features].copy()
    y_train = yf_df.loc[train_mask, target_column].copy()

    X_test = yf_df.loc[test_mask, model_features].copy()
    y_test = yf_df.loc[test_mask, target_column].copy()

    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ]
    )

    X_train_processed = fold_preprocessor.fit_transform(X_train)
    X_test_processed = fold_preprocessor.transform(X_test)

    fold_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_train_processed, y_train)
    y_pred = fold_model.predict(X_test_processed)

    fold_predictions = yf_df.loc[test_mask, [
        "SEASON_YEAR",
        "WEEK_INDEX"
    ]].copy()

    fold_predictions["ACTUAL"] = y_test.values
    fold_predictions["PREDICTED"] = y_pred
    fold_predictions["ABS_ERROR"] = (
        fold_predictions["ACTUAL"] -
        fold_predictions["PREDICTED"]
    ).abs()

    week_summary = (
        fold_predictions
        .groupby("WEEK_INDEX")["ABS_ERROR"]
        .agg(["count", "mean"])
        .reset_index()
    )

    week_summary["TEST_SEASON"] = fold["test_season"]

    week_results.append(week_summary)

week_results_df = pd.concat(
    week_results,
    ignore_index=True
)

week_results_df.rename(
    columns={"mean": "MAE"},
    inplace=True
)

week_results_df.sort_values(
    ["TEST_SEASON", "WEEK_INDEX"]
)

,WEEK_INDEX,count,MAE,TEST_SEASON
0,1,5,82.399591,2024
1,3,5,81.047662,2024
2,5,5,81.612546,2024
3,7,5,81.642545,2024
4,9,5,80.956699,2024
5,11,4,102.376177,2024
6,1,5,4.135036,2025
7,3,5,4.196547,2025
8,5,5,3.317099,2025
9,7,5,3.354068,2025


In [67]:
print(
    yf_df[
        ["SEASON_YEAR", "WEEK_INDEX"]
    ]
    .drop_duplicates()
    .sort_values(
        ["SEASON_YEAR", "WEEK_INDEX"]
    )
    .to_string(index=False)
)

 SEASON_YEAR  WEEK_INDEX
        2023           1
        2023           3
        2023           5
        2023           7
        2023           9
        2023          11
        2024           1
        2024           3
        2024           5
        2024           7
        2024           9
        2024          11
        2025           1
        2025           3
        2025           5
        2025           7
        2025           9
        2025          11
        2026           1
        2026           3
        2026           5
        2026           7
        2026           9
        2026          11


In [68]:
week_results = []

for fold in backtest_folds:
    train_mask = yf_df["SEASON_YEAR"].isin(fold["train_seasons"])
    test_mask = yf_df["SEASON_YEAR"] == fold["test_season"]

    X_train = yf_df.loc[train_mask, model_features].copy()
    y_train = yf_df.loc[train_mask, target_column].copy()

    X_test = yf_df.loc[test_mask, model_features].copy()
    y_test = yf_df.loc[test_mask, target_column].copy()

    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ]
    )

    X_train_processed = fold_preprocessor.fit_transform(X_train)
    X_test_processed = fold_preprocessor.transform(X_test)

    fold_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_train_processed, y_train)

    y_pred = fold_model.predict(X_test_processed)

    predictions = yf_df.loc[
        test_mask,
        ["SEASON_YEAR", "WEEK_INDEX"]
    ].copy()

    predictions["ACTUAL"] = y_test.values
    predictions["PREDICTED"] = y_pred
    predictions["ABS_ERROR"] = (
        predictions["ACTUAL"] -
        predictions["PREDICTED"]
    ).abs()

    week_summary = (
        predictions
        .groupby(
            ["SEASON_YEAR", "WEEK_INDEX"],
            as_index=False
        )
        .agg(
            COUNT=("ABS_ERROR", "size"),
            MAE=("ABS_ERROR", "mean")
        )
    )

    week_results.append(week_summary)

week_results_df = pd.concat(
    week_results,
    ignore_index=True
)

week_results_df.sort_values(
    ["SEASON_YEAR", "WEEK_INDEX"]
)

,SEASON_YEAR,WEEK_INDEX,COUNT,MAE
0,2024,1,5,82.399591
1,2024,3,5,81.047662
2,2024,5,5,81.612546
3,2024,7,5,81.642545
4,2024,9,5,80.956699
5,2024,11,4,102.376177
6,2025,1,5,4.135036
7,2025,3,5,4.196547
8,2025,5,5,3.317099
9,2025,7,5,3.354068


In [69]:
week_average = (
    week_results_df
    .groupby("WEEK_INDEX", as_index=False)
    .agg(
        SEASONS_EVALUATED=("SEASON_YEAR", "count"),
        MEAN_MAE=("MAE", "mean")
    )
    .sort_values("WEEK_INDEX")
)

week_average

,WEEK_INDEX,SEASONS_EVALUATED,MEAN_MAE
0,1,3,30.495286
1,3,3,30.185181
2,5,3,30.108672
3,7,3,29.905877
4,9,3,29.535418
5,11,3,36.867789


In [70]:
xgb_backtest_report = backtest_results_df[
    [
        "FOLD",
        "TRAIN_SEASONS",
        "TEST_SEASON",
        "TRAIN_ROWS",
        "TEST_ROWS",
        "MAE",
        "RMSE",
        "R2"
    ]
].copy()

xgb_backtest_report = xgb_backtest_report.round({
    "MAE": 3,
    "RMSE": 3,
    "R2": 3
})

xgb_backtest_report

,FOLD,TRAIN_SEASONS,TEST_SEASON,TRAIN_ROWS,TEST_ROWS,MAE,RMSE,R2
0,1,[2023],2024,30,29,84.407,106.087,-1.194
1,2,"[2023, 2024]",2025,59,30,3.650,4.425,0.996
2,3,"[2023, 2024, 2025]",2026,89,22,4.852,5.540,0.991


In [71]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

print("Hyperparameter search space:")
for parameter, values in param_grid.items():
    print(f"{parameter}: {values}")

Hyperparameter search space:
n_estimators: [100, 200, 300]
max_depth: [2, 3, 4]
learning_rate: [0.03, 0.05, 0.1]
subsample: [0.8, 1.0]
colsample_bytree: [0.8, 1.0]


In [72]:
def evaluate_xgb_params(params, folds):
    results = []

    for fold in folds:
        train_mask = yf_df["SEASON_YEAR"].isin(
            fold["train_seasons"]
        )
        test_mask = (
            yf_df["SEASON_YEAR"] == fold["test_season"]
        )

        X_train = yf_df.loc[
            train_mask, model_features
        ].copy()
        y_train = yf_df.loc[
            train_mask, target_column
        ].copy()

        X_test = yf_df.loc[
            test_mask, model_features
        ].copy()
        y_test = yf_df.loc[
            test_mask, target_column
        ].copy()

        fold_preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    SimpleImputer(strategy="median"),
                    numeric_features
                ),
                (
                    "categorical",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    categorical_features
                )
            ]
        )

        X_train_processed = (
            fold_preprocessor.fit_transform(X_train)
        )
        X_test_processed = (
            fold_preprocessor.transform(X_test)
        )

        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
            **params
        )

        model.fit(
            X_train_processed,
            y_train
        )

        y_pred = model.predict(
            X_test_processed
        )

        results.append({
            "TEST_SEASON": fold["test_season"],
            "MAE": mean_absolute_error(
                y_test, y_pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test, y_pred
                )
            ),
            "R2": r2_score(
                y_test, y_pred
            )
        })

    results_df = pd.DataFrame(results)

    return {
        "MEAN_MAE": results_df["MAE"].mean(),
        "MEAN_RMSE": results_df["RMSE"].mean(),
        "MEAN_R2": results_df["R2"].mean(),
        "FOLD_RESULTS": results_df
    }


print("Time-aware XGBoost tuning function created.")

Time-aware XGBoost tuning function created.


In [75]:
from itertools import product

tuning_results = []

param_names = list(param_grid.keys())
param_values = list(param_grid.values())

for values in product(*param_values):
    params = dict(zip(param_names, values))

    evaluation = evaluate_xgb_params(
        params,
        backtest_folds
    )

    tuning_results.append({
        **params,
        "MEAN_MAE": evaluation["MEAN_MAE"],
        "MEAN_RMSE": evaluation["MEAN_RMSE"],
        "MEAN_R2": evaluation["MEAN_R2"]
    })

tuning_results_df = pd.DataFrame(tuning_results)

print(
    "Completed configurations:",
    len(tuning_results_df)
)

tuning_results_df.head()

Completed configurations: 108


,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,MEAN_MAE,MEAN_RMSE,MEAN_R2
0,100,2,0.03,0.8,0.8,29.458367,36.592846,0.334024
1,100,2,0.03,0.8,1.0,31.849928,40.388631,0.165195
2,100,2,0.03,1.0,0.8,28.805764,36.323048,0.344213
3,100,2,0.03,1.0,1.0,31.918017,40.531112,0.161359
4,100,2,0.05,0.8,0.8,30.416907,38.107832,0.287902


In [77]:
best_xgb_configs = (
    tuning_results_df
    .sort_values(
        ["MEAN_MAE", "MEAN_RMSE"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

best_xgb_configs.head(10)

,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,MEAN_MAE,MEAN_RMSE,MEAN_R2
0,100,4,0.03,1.0,0.8,28.790671,36.526157,0.343850
1,100,3,0.03,1.0,0.8,28.802551,36.519491,0.343864
2,100,2,0.03,1.0,0.8,28.805764,36.323048,0.344213
3,100,4,0.03,0.8,0.8,29.411175,36.653111,0.332730
4,100,3,0.10,1.0,0.8,29.421706,36.549652,0.352619
5,100,3,0.03,0.8,0.8,29.424500,36.633716,0.332763
6,200,3,0.10,1.0,0.8,29.426080,36.554624,0.352565
7,300,3,0.10,1.0,0.8,29.426125,36.554641,0.352565
8,100,2,0.03,0.8,0.8,29.458367,36.592846,0.334024
9,100,4,0.10,1.0,0.8,29.471753,36.580433,0.352575


In [78]:
untuned_summary = {
    "MODEL": "Untuned XGBoost",
    "MEAN_MAE": backtest_results_df["MAE"].mean(),
    "MEAN_RMSE": backtest_results_df["RMSE"].mean(),
    "MEAN_R2": backtest_results_df["R2"].mean()
}

best_tuned = best_xgb_configs.iloc[0]

tuned_summary = {
    "MODEL": "Tuned XGBoost",
    "MEAN_MAE": best_tuned["MEAN_MAE"],
    "MEAN_RMSE": best_tuned["MEAN_RMSE"],
    "MEAN_R2": best_tuned["MEAN_R2"]
}

xgb_tuning_comparison = pd.DataFrame([
    untuned_summary,
    tuned_summary
])

xgb_tuning_comparison.round(3)

,MODEL,MEAN_MAE,MEAN_RMSE,MEAN_R2
0,Untuned XGBoost,30.969,38.684,0.264
1,Tuned XGBoost,28.791,36.526,0.344


In [79]:
best_params = {
    "n_estimators": int(best_tuned["n_estimators"]),
    "max_depth": int(best_tuned["max_depth"]),
    "learning_rate": float(best_tuned["learning_rate"]),
    "subsample": float(best_tuned["subsample"]),
    "colsample_bytree": float(best_tuned["colsample_bytree"])
}

tuned_evaluation = evaluate_xgb_params(
    best_params,
    backtest_folds
)

tuned_fold_results = tuned_evaluation["FOLD_RESULTS"]

print("Selected XGBoost parameters:")
print(best_params)

print("\nFold-level results:")
print(tuned_fold_results.round(3))

Selected XGBoost parameters:
{'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 1.0, 'colsample_bytree': 0.8}

Fold-level results:
   TEST_SEASON     MAE     RMSE     R2
0         2024  79.639  100.205 -0.958
1         2025   3.785    5.449  0.994
2         2026   2.948    3.925  0.995


In [80]:
model_comparison = pd.DataFrame([
    {
        "MODEL": "Historical Mean",
        "MEAN_MAE": np.nan,
        "MEAN_RMSE": np.nan,
        "MEAN_R2": np.nan
    },
    {
        "MODEL": "Previous Season Yield",
        "MEAN_MAE": ridge_backtest_results_df["MAE"].mean(),
        "MEAN_RMSE": ridge_backtest_results_df["RMSE"].mean(),
        "MEAN_R2": ridge_backtest_results_df["R2"].mean()
    },
    {
        "MODEL": "Untuned XGBoost",
        "MEAN_MAE": backtest_results_df["MAE"].mean(),
        "MEAN_RMSE": backtest_results_df["RMSE"].mean(),
        "MEAN_R2": backtest_results_df["R2"].mean()
    },
    {
        "MODEL": "Tuned XGBoost",
        "MEAN_MAE": tuned_fold_results["MAE"].mean(),
        "MEAN_RMSE": tuned_fold_results["RMSE"].mean(),
        "MEAN_R2": tuned_fold_results["R2"].mean()
    }
])

model_comparison.round(3)


,MODEL,MEAN_MAE,MEAN_RMSE,MEAN_R2
0,Historical Mean,NaN,NaN,NaN
1,Previous Season Yield,8.593,10.297,0.971
2,Untuned XGBoost,30.969,38.684,0.264
3,Tuned XGBoost,28.791,36.526,0.344


In [81]:
final_xgb_fold_results = tuned_fold_results.copy()

final_xgb_fold_results["MODEL"] = "Tuned XGBoost"

final_xgb_fold_results = final_xgb_fold_results[
    [
        "MODEL",
        "TEST_SEASON",
        "MAE",
        "RMSE",
        "R2"
    ]
].round(3)

final_xgb_fold_results

,MODEL,TEST_SEASON,MAE,RMSE,R2
0,Tuned XGBoost,2024,79.639,100.205,-0.958
1,Tuned XGBoost,2025,3.785,5.449,0.994
2,Tuned XGBoost,2026,2.948,3.925,0.995


In [82]:
final_xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

final_xgb_model.fit(
    X_processed,
    y
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


In [83]:
feature_importance = pd.DataFrame({
    "FEATURE": processed_feature_names,
    "IMPORTANCE": final_xgb_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("IMPORTANCE", ascending=False)
    .reset_index(drop=True)
)

print("Total processed features:", len(feature_importance))

feature_importance.head(20)

Total processed features: 77


,FEATURE,IMPORTANCE
0,numeric__PREVIOUS_SEASON_YIELD,0.722371
1,numeric__HISTORICAL_YIELD_STD,0.143941
2,numeric__HISTORICAL_MEAN_YIELD,0.132231
3,numeric__SDE_GDD_DEVIATION,0.000396
4,numeric__AVAILABLE_WEIGHT_PCT,0.000208
5,numeric__HISTORICAL_YIELD_TREND,0.000164
6,numeric__NDVI_STRESS_SCORE,0.000158
7,numeric__NDVI_DEVIATION,0.000128
8,numeric__WEATHER_DEVIATION,0.000125
9,numeric__HISTORICAL_YIELD_COUNT,0.000096


In [85]:
feature_importance_aggregated = []

for original_feature in model_features:
    matching = feature_importance[
        feature_importance["FEATURE"].str.contains(
            original_feature,
            regex=False
        )
    ]

    feature_importance_aggregated.append({
        "ORIGINAL_FEATURE": original_feature,
        "IMPORTANCE": matching["IMPORTANCE"].sum()
    })

feature_importance_aggregated = (
    pd.DataFrame(feature_importance_aggregated)
    .sort_values("IMPORTANCE", ascending=False)
    .reset_index(drop=True)
)

print(
    "Original model features:",
    len(feature_importance_aggregated)
)

feature_importance_aggregated.head(20)

Original model features: 57


,ORIGINAL_FEATURE,IMPORTANCE
0,PREVIOUS_SEASON_YIELD,0.722371
1,HISTORICAL_YIELD_STD,0.143941
2,HISTORICAL_MEAN_YIELD,0.132231
3,SDE_GDD_DEVIATION,0.000396
4,AVAILABLE_WEIGHT_PCT,0.000208
5,HISTORICAL_YIELD_TREND,0.000164
6,NDVI_STRESS_SCORE,0.000158
7,NDVI_DEVIATION,0.000128
8,WEATHER_DEVIATION,0.000125
9,HISTORICAL_YIELD_COUNT,0.000096


In [86]:
total_importance = (
    feature_importance_aggregated["IMPORTANCE"].sum()
)

print(
    "Total aggregated importance:",
    round(total_importance, 6)
)

print(
    "\nNumber of features with non-zero importance:",
    (
        feature_importance_aggregated["IMPORTANCE"] > 0
    ).sum()
)

print(
    "Number of features with zero importance:",
    (
        feature_importance_aggregated["IMPORTANCE"] == 0
    ).sum()
)

Total aggregated importance: 1.0

Number of features with non-zero importance: 13
Number of features with zero importance: 44


In [87]:
final_model_metadata = {
    "model_type": "XGBRegressor",
    "model_version": "xgboost_yield_forecast_v1",
    "xgboost_version": xgb.__version__,
    "target": target_column,
    "original_feature_count": len(model_features),
    "processed_feature_count": len(processed_feature_names),
    "numeric_feature_count": len(numeric_features),
    "categorical_feature_count": len(categorical_features),
    "numeric_imputation": "median",
    "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
    "n_estimators": 100,
    "max_depth": 4,
    "learning_rate": 0.03,
    "subsample": 1.0,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "validation_method": "expanding time-aware backtesting",
    "backtest_test_seasons": [2024, 2025, 2026],
    "mean_mae": float(tuned_fold_results["MAE"].mean()),
    "mean_rmse": float(tuned_fold_results["RMSE"].mean()),
    "mean_r2": float(tuned_fold_results["R2"].mean()),
    "uncertainty_method": "not yet defined"
}

print("Final model metadata created.")
print(final_model_metadata)

Final model metadata created.
{'model_type': 'XGBRegressor', 'model_version': 'xgboost_yield_forecast_v1', 'xgboost_version': '3.2.0', 'target': 'ACTUAL_YIELD_NUMERIC', 'original_feature_count': 57, 'processed_feature_count': 77, 'numeric_feature_count': 45, 'categorical_feature_count': 9, 'numeric_imputation': 'median', 'categorical_encoding': "OneHotEncoder(handle_unknown='ignore')", 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 1.0, 'colsample_bytree': 0.8, 'random_state': 42, 'validation_method': 'expanding time-aware backtesting', 'backtest_test_seasons': [2024, 2025, 2026], 'mean_mae': 28.790671376673775, 'mean_rmse': 36.52615734663393, 'mean_r2': 0.3438495980398235, 'uncertainty_method': 'not yet defined'}


In [88]:
all_backtest_errors = []

for fold in backtest_folds:
    train_mask = yf_df["SEASON_YEAR"].isin(
        fold["train_seasons"]
    )
    test_mask = (
        yf_df["SEASON_YEAR"] == fold["test_season"]
    )

    X_train = yf_df.loc[
        train_mask, model_features
    ].copy()
    y_train = yf_df.loc[
        train_mask, target_column
    ].copy()

    X_test = yf_df.loc[
        test_mask, model_features
    ].copy()
    y_test = yf_df.loc[
        test_mask, target_column
    ].copy()

    fold_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ]
    )

    X_train_processed = (
        fold_preprocessor.fit_transform(X_train)
    )
    X_test_processed = (
        fold_preprocessor.transform(X_test)
    )

    fold_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        max_depth=4,
        learning_rate=0.03,
        subsample=1.0,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(
        X_train_processed,
        y_train
    )

    y_pred = fold_model.predict(
        X_test_processed
    )

    errors = y_test.values - y_pred

    all_backtest_errors.extend(
        np.abs(errors)
    )

all_backtest_errors = np.array(
    all_backtest_errors
)

uncertainty_mae = np.mean(
    all_backtest_errors
)

uncertainty_p90 = np.percentile(
    all_backtest_errors,
    90
)

uncertainty_p95 = np.percentile(
    all_backtest_errors,
    95
)

print("Empirical absolute error:")
print("Mean :", round(uncertainty_mae, 3))
print("P90  :", round(uncertainty_p90, 3))
print("P95  :", round(uncertainty_p95, 3))

Empirical absolute error:
Mean : 30.715
P90  : 131.54
P95  : 131.54


In [89]:
final_model_metadata["uncertainty_method"] = (
    "Empirical absolute residual distribution "
    "from expanding time-aware backtesting"
)

final_model_metadata["uncertainty_mean_abs_error"] = round(
    uncertainty_mae, 3
)

final_model_metadata["uncertainty_p90_abs_error"] = round(
    uncertainty_p90, 3
)

final_model_metadata["uncertainty_p95_abs_error"] = round(
    uncertainty_p95, 3
)

print("Final model metadata updated.")
print(final_model_metadata)

Final model metadata updated.
{'model_type': 'XGBRegressor', 'model_version': 'xgboost_yield_forecast_v1', 'xgboost_version': '3.2.0', 'target': 'ACTUAL_YIELD_NUMERIC', 'original_feature_count': 57, 'processed_feature_count': 77, 'numeric_feature_count': 45, 'categorical_feature_count': 9, 'numeric_imputation': 'median', 'categorical_encoding': "OneHotEncoder(handle_unknown='ignore')", 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 1.0, 'colsample_bytree': 0.8, 'random_state': 42, 'validation_method': 'expanding time-aware backtesting', 'backtest_test_seasons': [2024, 2025, 2026], 'mean_mae': 28.790671376673775, 'mean_rmse': 36.52615734663393, 'mean_r2': 0.3438495980398235, 'uncertainty_method': 'Empirical absolute residual distribution from expanding time-aware backtesting', 'uncertainty_mean_abs_error': 30.715, 'uncertainty_p90_abs_error': 131.54, 'uncertainty_p95_abs_error': 131.54}
